# board_sync: camera board vs the engine's expected board (full design, perception)

README "Perception and rules": *detect mismatches between the physical board and the
engine's expected board, and recover*. The engine holds the authoritative game state; the
camera gives a `BoardPrediction` (`board_assembly.ipynb`). Before the robot moves, and after
the human moves, compare them:

* **match**: nothing to do.
* **one legal move away**: the human moved; accept it (M5's `infer_move` does this more
  robustly from probabilities).
* **anything else**: a piece was knocked over / misplaced, the robot dropped a piece, or the
  classifier is wrong. Then either ask the human to fix the board or (if the robot can) put
  the piece back itself. Which one is the policy decision left as TODO.

Milestone: full design (beyond the class project M1-M5).

**TODO**
- [ ] `decide_action`: the recovery policy (ask human vs auto-correct vs re-capture)
- [ ] use per-square confidence: a mismatch on a low-confidence square should first trigger a
      re-capture, not a correction
- [ ] hook into the game loop (owned by the game/engine package) once that exists

In [ ]:
from __future__ import annotations

from dataclasses import dataclass

import chess

%run ./labels.ipynb

## Differences

In [ ]:
@dataclass
class SquareDiff:
    square: str
    expected: str    # label on the engine board ("empty" or a FEN letter)
    observed: str    # label seen by the camera


def diff_boards(observed: chess.Board, expected: chess.Board) -> list[SquareDiff]:
    """Squares whose piece differs, in square-index order. Placement only."""
    o, e = board_to_labels(observed), board_to_labels(expected)
    return [SquareDiff(SQUARE_NAMES[sq], e[sq], o[sq]) for sq in chess.SQUARES if o[sq] != e[sq]]


def explaining_move(observed: chess.Board, expected: chess.Board) -> chess.Move | None:
    """A legal move from `expected` (with its side to move) whose result matches `observed`."""
    target = observed.board_fen()
    for move in expected.legal_moves:
        b = expected.copy(stack=False)
        b.push(move)
        if b.board_fen() == target:
            return move
    return None


def classify_mismatch(observed: chess.Board, expected: chess.Board) -> str:
    """"match", "legal_move" (one legal move away), "displaced" (same pieces, other squares),
    "missing" / "extra" (piece count lower / higher than expected) or "changed" (otherwise,
    e.g. a misclassified piece type)."""
    diffs = diff_boards(observed, expected)
    if not diffs:
        return "match"
    if explaining_move(observed, expected) is not None:
        return "legal_move"
    n_obs = sum(lab != "empty" for lab in board_to_labels(observed))
    n_exp = sum(lab != "empty" for lab in board_to_labels(expected))
    if n_obs < n_exp:
        return "missing"
    if n_obs > n_exp:
        return "extra"
    same_pieces = sorted(d.expected for d in diffs if d.expected != "empty") == \
        sorted(d.observed for d in diffs if d.observed != "empty")
    return "displaced" if same_pieces else "changed"

## Recovery policy (TODO)

In [ ]:
ACTIONS = ("ok", "accept_move", "recapture", "ask_human", "auto_correct")


def decide_action(observed: chess.Board, expected: chess.Board,
                  low_confidence: list[str] | None = None) -> str:
    """TODO: pick one of ACTIONS.

    Inputs: camera board, engine board, and the low-confidence squares of the BoardPrediction.
    Suggested policy to start from:
      match -> "ok"; legal_move -> "accept_move" (only when it is the human's turn);
      any differing square in low_confidence -> "recapture" (take another frame first);
      displaced with 1 piece and the robot just moved -> "auto_correct" (move it back);
      everything else -> "ask_human" (show the differing squares, wait for confirmation).
    Write down in the notes why; log how often each branch fires in real games.
    """
    raise NotImplementedError("TODO: decide_action policy")

## Self-check

In [ ]:
_exp = chess.Board()
assert classify_mismatch(chess.Board(), _exp) == "match" and diff_boards(chess.Board(), _exp) == []

_obs = chess.Board()
_obs.push_san("e4")
assert [d.square for d in diff_boards(_obs, _exp)] == ["e2", "e4"]
assert explaining_move(_obs, _exp) == chess.Move.from_uci("e2e4")
assert classify_mismatch(_obs, _exp) == "legal_move"

_obs = chess.Board()
_obs.remove_piece_at(chess.B1)
_obs.set_piece_at(chess.B3, chess.Piece.from_symbol("N"))   # knight moved illegally... b1-b3
assert classify_mismatch(_obs, _exp) == "displaced"

_obs = chess.Board()
_obs.remove_piece_at(chess.H2)
assert classify_mismatch(_obs, _exp) == "missing"
_obs = chess.Board()
_obs.set_piece_at(chess.E4, chess.Piece.from_symbol("p"))
assert classify_mismatch(_obs, _exp) == "extra"
_obs = chess.Board()
_obs.set_piece_at(chess.C1, chess.Piece.from_symbol("P"))   # bishop read as pawn
_d = diff_boards(_obs, _exp)
assert classify_mismatch(_obs, _exp) == "changed" and (_d[0].expected, _d[0].observed) == ("B", "P")
print("board_sync OK")